# IAM — Kaggle full-FT training (v0.19, Qwen3.5-2B on RTX 6000 Pro)

Full-parameter fine-tuning of **Qwen3.5-2B** for LLM-agent trace anomaly detection.

| Setting | Value |
|---|---|
| Base model | `/kaggle/input/models/barnobarno/qwen3.5-2b/transformers/unsloth/1` (Apache 2.0, Unsloth-prepared) |
| Fine-tune mode | **Full-parameter (no LoRA)** |
| Max seq len | 4096 |
| Hardware | **Kaggle RTX 6000 Pro (Blackwell, 96 GB)** |
| Training data | IAM v0.19: 13 attack families × DeepSeek-flash victim + ATBench + Agent3Sigma-Sweep + Scale AI MRT |
| Objectives | (a) next-action LM on action spans, (b) SFT on verdict span |
| Output | `/kaggle/working/submission_model/` (final) + `submission_model_best/` (best epoch) + `runs-eval/report/` |

## Kaggle inputs to attach (Add data)

1. **Dataset** `agent-iam-src` ← `dist/agent-iam-src.zip` (wheel + READMEs + this notebook)
2. **Dataset** `agent-iam-data` ← `dist/agent-iam-data.zip` (v0.19 tokenized JSONL)
3. **Model** `barnobarno/qwen3.5-2b` → variation `transformers/unsloth/1`
4. **Datasets** — the offline package wheel collection used by training-gg-0505 (Triton + unsloth/trl/etc.):
   - the Triton wheel(s) (e.g. `mayukh18/nemotron-packages` or your own mirror)
   - `ryanholbrook/nvidia_utility_script` → for `ptxas-blackwell` binary

Accelerator → **GPU RTX 6000 Pro** (Blackwell). Internet → optional (offline wheel path is the primary install route).

## 0. Mode selection

In [ ]:
# ============================================================
# MODE SELECTION — set exactly one to 1
# ============================================================
import os
import sys

os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: Full-FT training on Kaggle GPU
TRAIN_ON_KAGGLE = 1
# Mode B: Use pre-trained weights from a previous run and just repackage
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, "set exactly one mode"

# Unsloth-prepared base model checkpoint on Kaggle (Add data → Models →
# barnobarno/qwen3.5-2b → variation: transformers/unsloth/1).
# In transformers 5.x this loads as a Qwen3VLProcessor; cell 8 unwraps
# the inner text tokenizer so the rest of the pipeline is unchanged.
BASE_MODEL_ID   = "/kaggle/input/models/barnobarno/qwen3.5-2b/transformers/unsloth/1"
BASE_MODEL_NAME = "Qwen/Qwen3.5-2B"   # for adapter_config.base_model_name_or_path

MAX_SEQ_LEN     = 4096
LR              = 2e-5            # full-FT is more sensitive than LoRA — keep small
NUM_TRAIN_EPOCHS = 4
BATCH_SIZE      = 1               # peak memory governed by activations
GRAD_ACCUM      = 16              # effective batch = 16
SEED            = 42

OUT_DIR         = "/kaggle/working/sft_output"
SUBMISSION_DIR  = "/kaggle/working/submission_model"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "BASE_MODEL_ID":   BASE_MODEL_ID,
    "MAX_SEQ_LEN":     MAX_SEQ_LEN,
    "LR":              LR,
    "epochs":          NUM_TRAIN_EPOCHS,
    "effective_bs":    BATCH_SIZE * GRAD_ACCUM,
})


## 1. Install Unsloth + deps (offline-safe pattern from training-gg-0505)

In [ ]:
# Triton wheel (Blackwell-compatible). Mirrors training-gg-0505 cell 4.
import glob
import os
import site
import subprocess
import sys

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError(
        "No Triton wheel found under /kaggle/input. "
        "Attach a dataset that contains the Blackwell-compatible Triton wheel "
        "(e.g. mayukh18/nemotron-packages or a personal mirror)."
    )
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)

import importlib.util

print("triton spec:", importlib.util.find_spec("triton"))


In [ ]:
# ptxas-blackwell wiring — required for RTX 6000 Pro (Blackwell sm_120).
# Mirrors training-gg-0505 cell 5.
if TRAIN_ON_KAGGLE:
    import os
    import shutil
    import stat
    import sys

    # Utility script with the bundled ptxas-blackwell binary.
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Triton/ptxas-blackwell wiring applied.')
else:
    print("USE_PRETRAINED=1: skipping ptxas-blackwell setup.")


In [ ]:
# Offline install of unsloth/transformers/trl/peft/etc.
#
# Two wheel sources, both attached as Kaggle datasets:
#   1) nemotron-packages — base bundle (torch, triton, bitsandbytes, ...)
#      but its transformers (4.57.6) + unsloth are too old for Qwen3.5.
#   2) qwen35-wheels — built by notebooks/prep_qwen35_wheels.ipynb on a
#      CPU + Internet-ON kernel; contains transformers>=5.2.0, latest
#      unsloth + unsloth_zoo + bumped tokenizers/safetensors/hf_hub/...
#
# pip --no-index --find-links <A> --find-links <B> resolves across both
# dirs and picks the highest compatible version, so the new wheels in
# qwen35-wheels win over the older copies in nemotron-packages.
#
# No Internet needed at training time.
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import subprocess
    import sys

    import torch

    print("Python:", sys.version)
    print("Torch :", torch.__version__, "cuda=", torch.version.cuda)
    if not torch.cuda.is_available():
        raise RuntimeError("Full-FT requires a GPU runtime.")

    nemotron_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"

    # Auto-discover the qwen35 wheels dir by looking for the transformers
    # 5.x wheel itself — works regardless of what the user named the
    # Kaggle dataset slug or subfolder. e.g. matches both
    #   /kaggle/input/qwen35-wheels/transformers-5.2.0-...
    #   /kaggle/input/datasets/songningyu/qwen35whl/qwen35-wheels/transformers-5.2.0-...
    tf5_hits = sorted(glob.glob("/kaggle/input/**/transformers-5*.whl", recursive=True))
    if not tf5_hits:
        raise FileNotFoundError(
            "No transformers-5*.whl found under /kaggle/input. "
            "Run notebooks/prep_qwen35_wheels.ipynb on a CPU+Internet-ON kernel, "
            "publish its output as a Kaggle dataset, then attach it here."
        )
    qwen35_dir = os.path.dirname(tf5_hits[0])
    print("nemotron_dir =", nemotron_dir, "exists =", os.path.isdir(nemotron_dir))
    print("qwen35_dir   =", qwen35_dir)

    find_links_args = ["--find-links", qwen35_dir]
    if os.path.isdir(nemotron_dir):
        find_links_args += ["--find-links", nemotron_dir]

    # Install upgraded transformers + unsloth first so their pinned versions
    # win the resolve, then the rest of the stack.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--no-index", *find_links_args,
         "transformers>=5.2.0", "unsloth", "unsloth_zoo",
         "trl", "peft", "datasets", "accelerate", "bitsandbytes",
         "tokenizers", "safetensors", "huggingface_hub"],
        check=True,
    )

    # Install our own wheel from the agent-iam-src dataset.
    wheels = sorted(glob.glob("/kaggle/input/**/agent_iam-*.whl", recursive=True))
    if not wheels:
        raise FileNotFoundError("No agent_iam wheel under /kaggle/input — attach agent-iam-src dataset.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", wheels[-1]], check=True)
    print("Installed:", os.path.basename(wheels[-1]))

    # Verify versions actually picked up.
    import importlib.metadata as md
    for pkg in ("transformers", "unsloth", "unsloth_zoo", "trl", "peft", "tokenizers"):
        try:
            print(f"  {pkg}: {md.version(pkg)}")
        except md.PackageNotFoundError:
            print(f"  {pkg}: NOT INSTALLED")

    import transformers
    if transformers.__version__ < "5.2.0":
        raise RuntimeError(
            f"transformers={transformers.__version__} still too old. "
            "Re-run notebooks/prep_qwen35_wheels.ipynb and re-attach the dataset."
        )

    # IMPORTANT: if transformers/unsloth were previously imported in this
    # kernel session, restart the kernel (Run → Restart & Run All) before
    # continuing to cell 8, otherwise Python is still holding the old modules.


## 2. Load base model (Unsloth FastLanguageModel, **full-FT** mode)

In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    from unsloth import FastLanguageModel

    # Full-parameter fine-tuning: full_finetuning=True, no 4/8-bit quant,
    # no LoRA wrap. Unsloth still gives us fused kernels + gradient
    # checkpointing for memory savings.
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=True,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
        # Critical for Qwen3.5 stability: keep weights/grads in bf16 but
        # run softmax / layernorm / loss in fp32 to avoid NaN. Unsloth
        # explicitly flags this in the load-time warning for this model.
        float32_mixed_precision=True,
    )

    # Newer unsloth + transformers 5.x returns a Processor
    # (e.g. Qwen3VLProcessor) instead of a PreTrainedTokenizer. Keep a
    # reference to the processor for save-time, but unwrap to the inner
    # text tokenizer so .add_special_tokens / .pad_token_id work and so
    # downstream collator/SFTTrainer code stays unchanged.
    processor = tokenizer if hasattr(tokenizer, "tokenizer") else None
    if processor is not None:
        print(f"unwrapping {type(tokenizer).__name__} -> inner tokenizer")
        tokenizer = processor.tokenizer

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Add IAM special tokens and resize embeddings.
    from agent_iam.tokenize import SPECIAL_TOKENS
    added = tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
    print(f"added {added} special tokens; new vocab size = {len(tokenizer)}")
    if added > 0:
        # mean_resizing=True initializes new rows with the mean of existing
        # embedding rows instead of small random values — random init in
        # bf16 was producing degenerate rows that cascaded into NaN loss.
        try:
            model.resize_token_embeddings(len(tokenizer), mean_resizing=True)
        except TypeError:
            # older transformers without mean_resizing kw
            model.resize_token_embeddings(len(tokenizer))

    # Enable gradient checkpointing (use_reentrant=False for newer torch).
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

    n_params = sum(p.numel() for p in model.parameters())
    n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"model loaded. total={n_params/1e9:.2f}B  trainable={n_train/1e9:.2f}B "
          f"({100*n_train/n_params:.1f}%)")
else:
    print("USE_PRETRAINED=1: skipping base model load.")


## 3. Load the pre-tokenized v0.19 dataset

The Kaggle bundle's `agent-iam-data` dataset contains `train/val/test.tokenized.jsonl`,
each line `{id, input_ids, labels}` already encoded with the Qwen3.5 tokenizer +
our trajectory special tokens. The label region (`labels != -100`) is the verdict
content (symbol/type/reason inside `<verdict>...</verdict>`).

v0.19 source mix (2719 unique trajectories, 46% anomaly):
- 13 attack families generated with DeepSeek-flash victim at temp 0.5–0.7
  (mempoison3, tooldesc, mcp_rug_pull, iif3d2/iif3x2 verification-step, jwt_forgery, cross_tenant_leak, …)
- **Agent3Sigma-Sweep** (Apache 2.0, FIND-Lab × Ant Group): 213 cases
- ATBench (1000 real trajectories) + Scale AI MRT
- Train: 2157 (998 unsafe / 1159 safe) · Val: 293 · Test: 269


In [ ]:
if TRAIN_ON_KAGGLE:
    import glob
    import json
    import os

    from datasets import Dataset as HFDataset

    # Resolve the data root — supports both Kaggle mount layouts:
    #   /kaggle/input/<slug>/              (when attached as "datasets")
    #   /kaggle/input/datasets/<user>/<slug>/  (when attached with the
    #                                          datasets/<user>/<slug> prefix)
    candidates = [
        "/kaggle/input/datasets/songningyu/agent-iam-data",
        "/kaggle/input/agent-iam-data",
    ]
    DATA_ROOT = next((c for c in candidates if os.path.exists(c)), None)
    if DATA_ROOT is None:
        # last-resort recursive search
        hits = glob.glob("/kaggle/input/**/train.tokenized.jsonl", recursive=True)
        if not hits:
            raise FileNotFoundError(
                "no train.tokenized.jsonl found under /kaggle/input — "
                "did you attach the agent-iam-data dataset?"
            )
        DATA_ROOT = os.path.dirname(hits[0])
    print("DATA_ROOT =", DATA_ROOT)

    def load_split(name):
        path = f"{DATA_ROOT}/{name}.tokenized.jsonl"
        rows = []
        with open(path) as f:
            for line in f:
                d = json.loads(line)
                rows.append({
                    "input_ids":      d["input_ids"],
                    "labels":         d["labels"],
                    "attention_mask": [1] * len(d["input_ids"]),
                })
        print(f"  loaded {len(rows)} from {name}.tokenized.jsonl")
        return rows

    train_rows = load_split("train")
    # Optional: load val/test if you want eval during training
    # val_rows  = load_split("val")
    # test_rows = load_split("test")

    # ---- Dataset sanity + auto-filter ----------------------------------
    # All-(-100)-label rows divide-by-zero in CrossEntropy mean-reduction,
    # producing NaN loss for that micro-batch. With batch_size=1 +
    # grad_accum=16, even a single such row poisons the running loss
    # logger forever. Filter them out unconditionally and also report
    # vocab-range / length distribution so we can spot tokenizer bugs.
    _vocab_size = len(tokenizer)
    _all_ids   = [t for r in train_rows for t in r["input_ids"]]
    _all_lbls  = [t for r in train_rows for t in r["labels"]]
    _id_min, _id_max = min(_all_ids), max(_all_ids)
    _kept = sum(1 for l in _all_lbls if l != -100)
    _kept_per_row = [sum(1 for l in r["labels"] if l != -100) for r in train_rows]
    _len_per_row  = [len(r["input_ids"]) for r in train_rows]

    print(f"  input_ids: min={_id_min} max={_id_max} (tokenizer vocab={_vocab_size})")
    if _id_max >= _vocab_size:
        raise RuntimeError(
            f"input_ids contain token {_id_max} >= vocab {_vocab_size}. "
            "Either tokenizer wasn't resized for special tokens, or data was "
            "tokenized with a different tokenizer than the loaded model."
        )
    # Also check labels stay within vocab (or == -100)
    _lbl_max = max((l for l in _all_lbls if l != -100), default=-1)
    print(f"  labels   : kept={_kept}/{len(_all_lbls)} ({100*_kept/len(_all_lbls):.2f}%) "
          f"max-non-ignore={_lbl_max}")
    if _lbl_max >= _vocab_size:
        raise RuntimeError(
            f"labels contain id {_lbl_max} >= vocab {_vocab_size}. "
            "CrossEntropy will OOB and produce NaN."
        )

    # Dense-layout decision balance: count OK vs STOP verdict symbols in the
    # supervised labels. This is the imbalance W_STOP fights — if STOP is a
    # tiny fraction, the model will drift toward "always OK" without weighting.
    try:
        _ok_id = tokenizer.encode("OK", add_special_tokens=False)[0]
        _stop_id = tokenizer.encode("STOP", add_special_tokens=False)[0]
        _n_ok = sum(1 for l in _all_lbls if l == _ok_id)
        _n_stop = sum(1 for l in _all_lbls if l == _stop_id)
        _ratio = (_n_ok / _n_stop) if _n_stop else float("inf")
        print(f"  decisions: OK={_n_ok}  STOP={_n_stop}  (OK:STOP = {_ratio:.1f}:1) "
              f"-> a W_STOP near {_ratio:.0f} balances them")
    except Exception as _e:
        print(f"  [decision-balance probe skipped: {_e}]")

    _zero_rows = [i for i, k in enumerate(_kept_per_row) if k == 0]
    print(f"  rows w/ zero kept labels: {len(_zero_rows)}/{len(_kept_per_row)}")
    if _zero_rows:
        print(f"    first 10 such indices: {_zero_rows[:10]}")
        print("    -> FILTERING them out (they cause NaN loss in batch_size=1)")
        train_rows = [r for i, r in enumerate(train_rows) if i not in set(_zero_rows)]
        # Recompute length stats post-filter
        _len_per_row  = [len(r["input_ids"]) for r in train_rows]

    # Length distribution — extremely long rows can hit attention overflow
    # in linear-attn / mamba layers with bf16.
    _lp = sorted(_len_per_row)
    def _pct(p):
        return _lp[min(len(_lp)-1, int(len(_lp) * p))]
    print(f"  seq_len p50/p90/p95/p99/max: {_pct(0.5)}/{_pct(0.9)}/{_pct(0.95)}/{_pct(0.99)}/{_lp[-1]}")

    if _kept == 0:
        raise RuntimeError("All labels are -100 after filtering — re-check tokenization.")

    del _all_ids, _all_lbls

    hf_ds = HFDataset.from_list(train_rows).shuffle(seed=SEED)
    print(hf_ds)
    print("max input_ids len:", max(_len_per_row))
else:
    print("USE_PRETRAINED=1: skipping dataset load.")


## 4. Mode A: train

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob
    import os
    import time

    import torch
    from transformers import TrainerCallback
    from trl import SFTConfig, SFTTrainer

    # ---- Debug instrumentation toggles -----------------------------------
    # Training mechanism is confirmed healthy (overfit test passed 97.6% loss
    # drop, zero NaN across a full run), so the heavy diagnostic scaffolding
    # defaults OFF for speed + cleaner logs. Flip any to True to bring the
    # corresponding diagnostic back. The lightweight per-logging-step
    # loss/lr/grad_norm line, the per-epoch eval, and the loss-skip safety
    # net stay ON regardless.
    RUN_OVERFIT_TEST = False     # 50-step single-batch overfit sanity (~30s + 17GB transient)
    INSTALL_NAN_HOOKS = False    # 652 per-module forward NaN hooks (slow; only to localize a NaN)
    VERBOSE_MICROBATCH = False   # per-microbatch loss prints (first 32) — noisy
    NAN_WATCH_EVERY = 50         # NaNDetector watched-param print cadence (was 5)
    WATCH_PARAMS = False         # print embed/norm weight norms every NAN_WATCH_EVERY steps (NaN-debug aid)

    # ---- Dense-layout STOP weighting -------------------------------------
    # The dense per-step verdict format makes OK the dominant symbol (a trace
    # has N-1 OK decisions + 1 STOP). Up-weight the STOP symbol token in the
    # loss so the model doesn't collapse to "always OK" (under-flagging).
    # Set to 1.0 to disable. Tune (5/10/15) against attack_stop@anomaly vs
    # benign_stop in the per-epoch eval.
    W_STOP = 8.0

    # ---- Per-epoch eval callback ----------------------------------------
    # After every full epoch, score test.jsonl with the in-memory model
    # and write per-epoch scores.jsonl. The keyword baseline is computed
    # once (it doesn't change between epochs). This lets us see the
    # PR@1% trajectory across training instead of only the final number.
    class PerEpochEvalCallback(TrainerCallback):
        def __init__(self, eval_root, tokenizer, threshold=8.0,
                     best_dir=None, target_fsr=0.01, processor=None):
            self.eval_root = eval_root
            self.tokenizer = tokenizer
            self.threshold = threshold
            self.best_dir = best_dir
            self.target_fsr = target_fsr   # primary metric: PR @ FSR <= target_fsr
            self.processor = processor      # save processor with best ckpt if provided
            self._test_path = None
            self._keyword_done = False
            self._best_pr = float("-inf")
            self._best_epoch = None

        def _find_test(self):
            if self._test_path:
                return self._test_path
            hits = sorted(glob.glob("/kaggle/input/**/test.jsonl", recursive=True))
            hits = [p for p in hits if not p.endswith(".tokenized.jsonl")]
            self._test_path = hits[0] if hits else None
            return self._test_path

        def on_epoch_end(self, args, state, control, model=None, **kwargs):
            test_path = self._find_test()
            if test_path is None:
                print("[PerEpochEval] no test.jsonl under /kaggle/input — skipping eval")
                return control

            from agent_iam.detect.online import TraceMonitor
            from agent_iam.eval.baselines import keyword_scorer
            from agent_iam.eval.runner import monitor_scorer, run_split, verdict_scorer

            epoch = int(round(state.epoch))
            epoch_dir = os.path.join(self.eval_root, f"epoch-{epoch}")
            os.makedirs(epoch_dir, exist_ok=True)

            print(f"\n[PerEpochEval] epoch={epoch}: scoring on {test_path}")
            was_training = model.training
            model.eval()
            try:
                monitor = TraceMonitor(model=model, tokenizer=self.tokenizer, threshold=self.threshold)

                # ---- 1) verdict-head scoring (the design we just shipped) ----
                # One attack -> two queries: POS_ANOMALY (expect STOP) and
                # POS_ATTACK_SAFE (expect NOT STOP). One benign -> POS_BENIGN
                # at a seeded-random step (expect NOT STOP). Matches the
                # training-time verdict supervision.
                t0 = time.time()
                summary = run_split(
                    verdict_scorer(monitor, seed=42),
                    test_path,
                    f"{epoch_dir}/scores.jsonl",
                    progress_every=50,
                )
                print(f"  -> verdict scorer: {summary['traces']} traces in {time.time()-t0:.1f}s")

                # ---- 2) action-PPL baseline scorer for comparison ----
                # Old eval method; runs alongside so we can quantify how
                # much the verdict-head approach actually helps. This used
                # to be the only path and reported PR@1%=0 in every epoch.
                t0b = time.time()
                run_split(
                    monitor_scorer(monitor),
                    test_path,
                    f"{epoch_dir}/scores_baseline_ppl.jsonl",
                    progress_every=50,
                )
                print(f"  -> ppl baseline:   in {time.time()-t0b:.1f}s")

                # ---- 3) Compute metrics ----
                from agent_iam.eval.metrics import (
                    pr_at_fsr,
                    query_accuracy,
                    query_auroc,
                    query_pr_at_fsr,
                    query_pstop_means,
                    reason_similarity,
                    sweep,
                )
                from agent_iam.eval.runner import load_results
                _res = load_results(f"{epoch_dir}/scores.jsonl")
                _res_ppl = load_results(f"{epoch_dir}/scores_baseline_ppl.jsonl")

                # Position-granularity 0/1 accuracy (the new headline numbers)
                _qa = query_accuracy(_res)
                print(f"  -> queries: anomaly={_qa.n_anomaly} "
                      f"attack_safe={_qa.n_attack_safe} benign={_qa.n_benign}")
                print(f"  -> attack_stop@anomaly = {_qa.attack_stop_at_anomaly:.3f}  "
                      f"(TP — should be high)")
                print(f"  -> attack_stop@safe    = {_qa.attack_stop_at_safe:.3f}  "
                      f"(in-trace FP — should be low)")
                print(f"  -> benign_stop         = {_qa.benign_stop:.3f}  "
                      f"(classic FSR — should be low)")
                print(f"  -> overall_symbol_acc  = {_qa.overall_acc:.3f}")

                # Continuous PR/FSR sweep on query-level p_stop
                _qop = query_pr_at_fsr(_res, target_fsr=self.target_fsr)
                pr = _qop.pr if (_qop.pr == _qop.pr) else float("-inf")
                print(f"  -> query PR @ FSR<={self.target_fsr:.0%}: "
                      f"{_qop.pr:.3f} (threshold p_stop={_qop.threshold})")
                # AUROC + mean p_stop per query type — discrimination/separation,
                # independent of threshold (rises even when PR@1% is still 0).
                _auc = query_auroc(_res)
                _pm = query_pstop_means(_res)
                print(f"  -> query AUROC: {_auc:.3f}")
                print(f"  -> mean p_stop: anomaly={_pm.get('anomaly', float('nan')):.3f}  "
                      f"attack_safe={_pm.get('attack_safe', float('nan')):.3f}  "
                      f"benign={_pm.get('benign_random', float('nan')):.3f}")

                # Legacy step-level PR/FSR on the SAME verdict scores
                # (for comparison with prior runs' numbers)
                # Legacy step-based sweep on a probability-scale grid.
                # (default thresholds run 0.5..100 in PPL units — useless for
                # 0..1 probabilities; we override with a 0..1 grid here.)
                _prob_grid = [round(i * 0.01, 4) for i in range(101)]
                _op_legacy = pr_at_fsr(sweep(_res, thresholds=_prob_grid), self.target_fsr)
                print(f"  -> [legacy step] PR @ FSR<={self.target_fsr:.0%}: "
                      f"{_op_legacy.pr:.3f} (threshold={_op_legacy.threshold})")

                # PPL baseline PR/FSR (legacy method, for direct comparison)
                _op_ppl = pr_at_fsr(sweep(_res_ppl), self.target_fsr)
                print(f"  -> [ppl baseline] PR @ FSR<={self.target_fsr:.0%}: "
                      f"{_op_ppl.pr:.3f} (threshold={_op_ppl.threshold})")

                # Per-family breakdown of attack_stop@anomaly
                if _qa.attack_stop_at_anomaly_per_family:
                    _by_fam = sorted(_qa.attack_stop_at_anomaly_per_family.items(),
                                     key=lambda kv: -kv[1])
                    print("  -> attack_stop@anomaly by family (top 10):")
                    for _fam, _v in _by_fam[:10]:
                        print(f"       {_fam:<14} {_v:.3f}")
                if _qa.benign_stop_per_source:
                    print("  -> benign_stop by source:")
                    for _src, _v in sorted(_qa.benign_stop_per_source.items()):
                        print(f"       {_src:<14} {_v:.3f}")

                # Reason semantic similarity (only POS_ANOMALY + STOP)
                try:
                    _sim = reason_similarity(_res, embedder=monitor.embed_text)
                    print(f"  -> reason cosine sim: {_sim.mean_cosine:.3f} "
                          f"(scored {_sim.n_scored}/{_sim.n_eligible} eligible attacks)")
                    if _sim.per_family:
                        _by_fam = sorted(_sim.per_family.items(), key=lambda kv: -kv[1])
                        print("  -> reason cos by family (top 8):")
                        for _fam, _c in _by_fam[:8]:
                            print(f"       {_fam:<14} {_c:.3f}")
                except Exception as _se:
                    _sim = None
                    print(f"  [PerEpochEval] reason_similarity failed: {_se}")

                # Persist a full per-epoch summary alongside scores.jsonl
                import json as _json
                with open(f"{epoch_dir}/eval_summary.json", "w") as _f:
                    _json.dump({
                        "epoch": epoch,
                        "queries": {
                            "n_anomaly": _qa.n_anomaly,
                            "n_attack_safe": _qa.n_attack_safe,
                            "n_benign": _qa.n_benign,
                        },
                        "symbol_acc": {
                            "attack_stop_at_anomaly": _qa.attack_stop_at_anomaly,
                            "attack_stop_at_safe": _qa.attack_stop_at_safe,
                            "benign_stop": _qa.benign_stop,
                            "overall": _qa.overall_acc,
                        },
                        "query_auroc": _auc,
                        "mean_p_stop": _pm,
                        "pr_at_fsr": {
                            "target_fsr": self.target_fsr,
                            "query_based": {
                                "pr": _qop.pr,
                                "threshold": _qop.threshold,
                                "fsr": _qop.fsr,
                            },
                            "legacy_step_based": {
                                "pr": _op_legacy.pr,
                                "threshold": _op_legacy.threshold,
                            },
                            "ppl_baseline": {
                                "pr": _op_ppl.pr,
                                "threshold": _op_ppl.threshold,
                            },
                        },
                        "reason_similarity": (None if _sim is None else {
                            "n_eligible": _sim.n_eligible,
                            "n_scored": _sim.n_scored,
                            "mean_cosine": _sim.mean_cosine,
                            "per_family": _sim.per_family,
                        }),
                        "per_family_attack_stop_at_anomaly": _qa.attack_stop_at_anomaly_per_family,
                        "per_source_benign_stop": _qa.benign_stop_per_source,
                    }, _f, indent=2)

                # Save best-so-far model.
                # First epoch: always save (so there's a fallback checkpoint
                # even when PR is NaN, e.g. no threshold satisfies FSR cap).
                # Subsequent epochs: only when PR strictly improves.
                _is_first = (self._best_epoch is None)
                _improves = (pr > self._best_pr)
                if self.best_dir is not None and (_is_first or _improves):
                    import json as _json
                    print(f"  -> NEW BEST (prev={self._best_pr if self._best_epoch else 'n/a'}); saving to {self.best_dir}")
                    os.makedirs(self.best_dir, exist_ok=True)
                    model.save_pretrained(
                        self.best_dir,
                        safe_serialization=True,
                        max_shard_size="5GB",
                    )
                    if self.processor is not None:
                        self.processor.save_pretrained(self.best_dir)
                    else:
                        self.tokenizer.save_pretrained(self.best_dir)
                    with open(f"{self.best_dir}/best_epoch.json", "w") as _f:
                        _json.dump({
                            "epoch": epoch,
                            # NOTE: _qop is the query-based operating point
                            # (replaces old PPL-based _op variable).
                            "pr_at_fsr_001": _qop.pr,
                            "threshold": _qop.threshold,
                            "fsr_actual": _qop.fsr,
                            "scores_jsonl": f"{epoch_dir}/scores.jsonl",
                            "symbol_acc": {
                                "attack_stop_at_anomaly": _qa.attack_stop_at_anomaly,
                                "attack_stop_at_safe": _qa.attack_stop_at_safe,
                                "benign_stop": _qa.benign_stop,
                            },
                        }, _f, indent=2)
                    self._best_pr = pr
                    self._best_epoch = epoch
            except Exception as e:
                print(f"  [PerEpochEval] eval failed: {e}")
            finally:
                if was_training:
                    model.train()

            if not self._keyword_done:
                kw_dir = os.path.join(self.eval_root, "keyword")
                os.makedirs(kw_dir, exist_ok=True)
                run_split(keyword_scorer(), test_path, f"{kw_dir}/scores.jsonl", progress_every=200)
                self._keyword_done = True
                print("  -> keyword baseline scored once for the run")

            return control

    # ---- Per-step NaN / weight-blow-up detector -------------------------
    # Fires every optimizer step. Records grad norms, then after the step
    # checks if any weight became non-finite. On first trip it dumps the
    # offending params + the top grad norms from the step that killed
    # them, and signals the trainer to stop. This isolates "which layer
    # blew up first" — almost always embed_tokens / lm_head rows for the
    # newly-added special tokens with a too-aggressive LR.
    class NaNDetector(TrainerCallback):
        def __init__(self, watch_substrings=("embed_tokens", "lm_head",
                                              "layers.0.linear_attn.norm",
                                              "layers.11.post_attention_layernorm",
                                              "layers.23.post_attention_layernorm"),
                     top_k=10, log_every=5):
            self.watch = watch_substrings
            self.top_k = top_k
            self.log_every = log_every
            self._last_grad_norms = {}
            self._last_nonfinite_grads = []
            self._tripped = False

        def _watched(self, model):
            out = []
            with torch.no_grad():
                for n, p in model.named_parameters():
                    if any(s in n for s in self.watch):
                        t = p.detach().float()
                        out.append((n, t.norm().item(), t.abs().max().item()))
            return out

        def _bad_weights(self, model):
            bad = []
            for n, p in model.named_parameters():
                t = p.detach()
                if torch.isnan(t).any().item():
                    bad.append((n, "nan", tuple(t.shape)))
                elif torch.isinf(t).any().item():
                    bad.append((n, "inf", tuple(t.shape)))
            return bad

        def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
            if self._tripped or model is None:
                return control
            g = []
            for n, p in model.named_parameters():
                if p.grad is None:
                    continue
                gn = p.grad.detach().float().norm().item()
                g.append((n, gn))
            self._last_grad_norms = dict(g)
            self._last_nonfinite_grads = [
                (n, gn) for n, gn in g
                if (gn != gn) or gn == float("inf") or gn == float("-inf")
            ]
            if self._last_nonfinite_grads:
                print(f"\n[NaNDetector] step={state.global_step} "
                      f"non-finite GRAD on {len(self._last_nonfinite_grads)} params:")
                for n, gn in self._last_nonfinite_grads[:self.top_k]:
                    print(f"    grad={gn!r:>12}  {n}")
            return control

        def on_step_end(self, args, state, control, model=None, logs=None, **kwargs):
            if self._tripped or model is None:
                return control

            if WATCH_PARAMS and (state.global_step <= 3 or state.global_step % self.log_every == 0):
                print(f"[watch] step={state.global_step} param norms:")
                for n, wn, mx in self._watched(model):
                    print(f"    norm={wn:14.6g}  max|w|={mx:14.6g}  {n}")

            bad = self._bad_weights(model)
            if bad:
                print(f"\n[NaNDetector] step={state.global_step} "
                      f"non-finite WEIGHTS on {len(bad)} params:")
                for n, kind, shp in bad[:self.top_k]:
                    print(f"    {kind}  shape={shp}  {n}")
                top = sorted(
                    self._last_grad_norms.items(),
                    key=lambda x: -(x[1] if x[1] == x[1] else 0.0),
                )[:self.top_k]
                print(f"  top {self.top_k} grad norms on the step that killed them:")
                for n, gn in top:
                    print(f"    {gn:12.4g}  {n}")
                self._tripped = True
                control.should_training_stop = True
            return control

        def on_log(self, args, state, control, logs=None, **kwargs):
            if not logs:
                return control
            loss = logs.get("loss")
            lr = logs.get("learning_rate")
            gn = logs.get("grad_norm")
            if loss is not None:
                msg = f"[train] step={state.global_step} loss={loss}"
                if lr is not None: msg += f" lr={lr:.2g}"
                if gn is not None: msg += f" grad_norm={gn:.4g}"
                print(msg)
                if (loss != loss) or loss in (float("inf"), float("-inf")):
                    print("  ^^ NON-FINITE LOSS")
            return control

    # Padding collator: pad input_ids with pad_token_id, labels with -100.
    def collator(batch):
        max_l = max(len(b["input_ids"]) for b in batch)
        pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
        def pad(seq, value):
            return seq + [value] * (max_l - len(seq))
        return {
            "input_ids":      torch.tensor([pad(b["input_ids"], pad_id) for b in batch]),
            "labels":         torch.tensor([pad(b["labels"], -100)     for b in batch]),
            "attention_mask": torch.tensor([
                [1] * len(b["input_ids"]) + [0] * (max_l - len(b["input_ids"]))
                for b in batch
            ]),
        }

    training_args = SFTConfig(
        output_dir=OUT_DIR,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        # full-FT-friendly Adam betas (per gg-0505)
        adam_beta1=0.9,
        adam_beta2=0.95,
        adam_epsilon=1e-8,
        weight_decay=0.0,
        max_grad_norm=10.0,   # was 1.0; global clip was dominated by embed_tokens grad (77) and zeroed body updates
        logging_steps=10,
        # No intermediate checkpoints — we only need the final weights at end of training.
        save_strategy="no",
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=MAX_SEQ_LEN,
        dataset_kwargs={"skip_prepare_dataset": True},   # already tokenized
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=SEED,
        report_to="none",
        packing=False,
        # Use fused AdamW since we have 96GB of room
        optim="adamw_torch_fused",
    )

    # ---- Per-microbatch loss probe + forward NaN tracer + loss-skip -----
    # Three things at once:
    #   (a) Forward hooks on every module record where the FIRST NaN /
    #       Inf appears in the activation flow on a bad micro-batch.
    #       Tells us "the NaN was born in model.language_model.layers.5.
    #       linear_attn" instead of just "the loss is NaN".
    #   (b) On non-finite loss we dump the offending sample (shape,
    #       label count, head/tail input ids, first-NaN module) and then
    #       REPLACE the loss with a zero-tensor anchored to a model
    #       parameter so the running-loss accumulator is no longer
    #       poisoned and the trainer keeps progressing on the good
    #       batches. Without this, one bad micro-batch makes every
    #       subsequent logged loss NaN forever.
    #   (c) Forward hooks have a small bool flag so we don't do the
    #       isfinite() scan every step in steady state — only when we
    #       expect a bad batch (always, for now, but cheap on GPU).
    class LossSanityTrainer(SFTTrainer):
        _bad_microbatches = 0
        _max_dump = 8
        _hooks_installed = False
        _first_nan = None   # (module_name, type_name, shape, n_nan, n_inf)
        _check_activations = True

        @staticmethod
        def _scan_tensor(t):
            if not isinstance(t, torch.Tensor) or not t.dtype.is_floating_point:
                return 0, 0
            n_nan = int(torch.isnan(t).sum().item())
            n_inf = int(torch.isinf(t).sum().item())
            return n_nan, n_inf

        @classmethod
        def _install_nan_hooks(cls, model):
            # Gated: the 652 per-module forward hooks add ~50s/epoch of GPU
            # syncs and are only needed to localize *where* a NaN is born.
            if not INSTALL_NAN_HOOKS:
                return
            if cls._hooks_installed:
                return
            cls._hooks_installed = True
            def make_hook(name, type_name):
                def hook(module, inp, out):
                    # Skip scan when:
                    #   - explicitly disabled
                    #   - already tripped
                    #   - module is in eval mode (PerEpochEvalCallback wraps
                    #     scoring in model.eval(); skipping here saves ~650
                    #     GPU syncs per forward pass during eval)
                    if (not cls._check_activations
                            or cls._first_nan is not None
                            or not module.training):
                        return
                    tensors = []
                    if isinstance(out, torch.Tensor):
                        tensors = [out]
                    elif isinstance(out, (tuple, list)):
                        tensors = [t for t in out if isinstance(t, torch.Tensor)]
                    elif hasattr(out, "last_hidden_state"):
                        tensors = [out.last_hidden_state]
                    for t in tensors:
                        n_nan, n_inf = cls._scan_tensor(t)
                        if n_nan or n_inf:
                            cls._first_nan = (name, type_name, tuple(t.shape), n_nan, n_inf)
                            return
                return hook
            n_installed = 0
            for n, m in model.named_modules():
                if n == "":  # skip root
                    continue
                # Skip pure containers (nn.ModuleList etc.) - they don't have
                # meaningful forward outputs; their children do.
                if type(m).__name__ in ("ModuleList", "ModuleDict", "Sequential"):
                    continue
                m.register_forward_hook(make_hook(n, type(m).__name__))
                n_installed += 1
            print(f"  [LossSanity] installed {n_installed} forward NaN hooks")

        _total_microbatches = 0
        def compute_loss(self, model, inputs, return_outputs=False,
                         num_items_in_batch=None):
            LossSanityTrainer._install_nan_hooks(model)
            LossSanityTrainer._first_nan = None
            out = super().compute_loss(
                model, inputs,
                return_outputs=True,
                num_items_in_batch=num_items_in_batch,
            )
            loss, outputs = out if isinstance(out, tuple) else (out, None)
            LossSanityTrainer._total_microbatches += 1
            # First 32 micro-batches: print actual loss value so we can SEE
            # whether the trainer is consuming finite, varying loss values.
            if VERBOSE_MICROBATCH and LossSanityTrainer._total_microbatches <= 32:
                try:
                    _lv = float(loss.detach().item())
                    print(f"  [LossSanity] microbatch #{LossSanityTrainer._total_microbatches}: loss={_lv:.4f}")
                except Exception:
                    pass
            if not torch.isfinite(loss).all():
                LossSanityTrainer._bad_microbatches += 1
                idx = LossSanityTrainer._bad_microbatches
                if idx <= LossSanityTrainer._max_dump:
                    labels = inputs.get("labels")
                    ids = inputs.get("input_ids")
                    kept = int((labels != -100).sum().item()) if labels is not None else -1
                    seq_len = ids.shape[1] if ids is not None else -1
                    bs = ids.shape[0] if ids is not None else -1
                    print(f"\n[LossSanity] BAD #{idx}: loss={loss.item()} "
                          f"shape=({bs},{seq_len}) kept_labels={kept}")
                    fn = LossSanityTrainer._first_nan
                    if fn is not None:
                        print(f"  first NaN/Inf: module={fn[0]} ({fn[1]}) "
                              f"shape={fn[2]} n_nan={fn[3]} n_inf={fn[4]}")
                    else:
                        print("  no module output had NaN — loss-fn issue "
                              "(likely 0/0 from all-(-100) labels, or "
                              "upcast-to-fp32 happened only inside loss)")
                    if labels is not None and kept > 0:
                        lb = labels[0].tolist()
                        kept_idx = [i for i, x in enumerate(lb) if x != -100]
                        print(f"  kept indices (first 10): {kept_idx[:10]}")
                        print(f"  kept values  (first 10): {[lb[i] for i in kept_idx[:10]]}")
                    if ids is not None:
                        inp = ids[0].tolist()
                        print(f"  input head: {inp[:20]}")
                        print(f"  input tail: {inp[-20:]}")
                    if outputs is not None and hasattr(outputs, "logits") and outputs.logits is not None:
                        lg = outputs.logits.detach().float()
                        print(f"  logits: shape={tuple(lg.shape)} "
                              f"min={lg.min().item():.3g} max={lg.max().item():.3g} "
                              f"any_nan={torch.isnan(lg).any().item()} "
                              f"any_inf={torch.isinf(lg).any().item()}")
                # ----- LOSS SKIP -----
                # Replace with zero tensor anchored to a model parameter so
                # backward still has a graph (zero grad on the anchor param,
                # which is harmless) and the running-loss accumulator gets
                # 0.0 instead of NaN. Training continues on subsequent
                # batches; the optimizer just sees a wasted step here.
                anchor = next(p for p in model.parameters() if p.requires_grad)
                loss = (anchor.float().sum() * 0.0).to(anchor.dtype)

            # ----- STOP-weighted re-loss (dense layout class imbalance) -----
            # Recompute CE from logits with a higher weight on the STOP symbol
            # token, so OK-dominant dense supervision doesn't push the model to
            # never flag. Skipped when W_STOP==1.0 or after a loss-skip.
            if (W_STOP != 1.0 and outputs is not None
                    and getattr(outputs, "logits", None) is not None
                    and torch.isfinite(loss).all()):
                _labels = inputs.get("labels")
                if _labels is not None:
                    if not hasattr(LossSanityTrainer, "_stop_token_id"):
                        LossSanityTrainer._stop_token_id = self.processing_class.encode(
                            "STOP", add_special_tokens=False
                        )[0]
                    _stop_id = LossSanityTrainer._stop_token_id
                    if not hasattr(LossSanityTrainer, "_weight_announced"):
                        LossSanityTrainer._weight_announced = True
                        print(f"  [LossSanity] STOP-weighted CE ACTIVE: "
                              f"W_STOP={W_STOP}, stop_token_id={_stop_id}")
                    _sl = outputs.logits[:, :-1, :].float()      # next-token shift
                    _tl = _labels[:, 1:].reshape(-1)
                    _ce = torch.nn.functional.cross_entropy(
                        _sl.reshape(-1, _sl.size(-1)), _tl,
                        ignore_index=-100, reduction="none",
                    )
                    _w = torch.ones_like(_ce)
                    _w[_tl == _stop_id] = W_STOP
                    _keep = (_tl != -100).float()
                    _denom = (_w * _keep).sum().clamp(min=1.0)
                    loss = (_ce * _w * _keep).sum() / _denom

            return (loss, outputs) if return_outputs else loss

    trainer = LossSanityTrainer(
        model=model,
        args=training_args,
        train_dataset=hf_ds,
        processing_class=tokenizer,
        data_collator=collator,
        callbacks=[
            PerEpochEvalCallback(
                eval_root="/kaggle/working/runs-eval",
                tokenizer=tokenizer,
                processor=(processor if 'processor' in dir() else None),
                # inf -> skip per-step verdict text generation; only need
                # PPL scores for PR/FSR metric. Brings eval from ~55 min
                # to ~3 min for 269 test traces.
                threshold=float("inf"),
                best_dir="/kaggle/working/submission_model_best",
                target_fsr=0.01,
            ),
            NaNDetector(log_every=NAN_WATCH_EVERY),
        ],
    )

    if RUN_OVERFIT_TEST:
        # =====================================================================
        # OVERFIT TEST — the actual "is training mechanism working?" check.
        #
        # Take ONE batch, run forward+backward+optimizer.step 50 times. If loss
        # drops from ~5 to <1, the training mechanism is fine (optimizer steps,
        # autograd, autocast, Unsloth grad offload — everything wired up).
        # If loss is flat, something is broken regardless of what trainer.train()
        # reports.
        #
        # We snapshot all trainable params before the test and restore them
        # after, so the real training starts from base-model weights (not from
        # an overfit checkpoint on one batch).
        # =====================================================================
        print("=" * 60)
        print("OVERFIT SANITY TEST: 50 steps on one batch")
        print("=" * 60)

        _it = iter(trainer.get_train_dataloader())
        _batch = next(_it)
        _batch = {k: v.to(model.device) for k, v in _batch.items() if hasattr(v, "to")}
        model.train()

        # Snapshot trainable params (so we can restore base weights afterward)
        print("  snapshotting trainable params for post-test restore...")
        import time as _time
        _t_snap = _time.time()
        _snap = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        print(f"  snapshotted {len(_snap)} params in {_time.time()-_t_snap:.1f}s")

        # Pick a body param to track (something deep in the body — not embed/lm_head)
        # Try mid-layers first; fall back to any 2D body weight so the test
        # still runs even if the layer naming convention differs.
        def _pick_body_param():
            for _layer_pat in ("layers.11", "layers.12", "layers.10", "layers.0"):
                for _n, _p in model.named_parameters():
                    if (_p.requires_grad and _p.ndim >= 2
                            and "embed" not in _n and "lm_head" not in _n
                            and _layer_pat in _n
                            and "norm" not in _n):
                        return _n, _p
            # Final fallback: any 2D trainable weight that isn't embed/head
            for _n, _p in model.named_parameters():
                if (_p.requires_grad and _p.ndim >= 2
                        and "embed" not in _n and "lm_head" not in _n
                        and "norm" not in _n):
                    return _n, _p
            return None, None

        _body_name, _body_param = _pick_body_param()
        if _body_param is None:
            raise RuntimeError("Couldn't find a body weight to track. Model has no 2D non-embed/head params?")
        _body_before = _body_param.detach().clone()
        print(f"  tracking body weight: {_body_name}")
        print(f"    initial: norm={_body_before.float().norm().item():.6g} "
              f"max|w|={_body_before.float().abs().max().item():.6g}")
        print(f"    device={_body_param.device}  dtype={_body_param.dtype}  "
              f"requires_grad={_body_param.requires_grad}")
        print(f"  model.training = {model.training}  "
              f"(must be True or dropout/norm wrong)")
        # also report the batch tensor placement so we know they're co-located
        print(f"  batch input_ids: device={_batch['input_ids'].device} "
              f"shape={tuple(_batch['input_ids'].shape)}")

        # Fresh AdamW (mirrors trainer settings; not Unsloth-patched so we
        # isolate "does the optimizer mechanism update weights at all").
        _test_opt = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
        )

        _losses = []
        _printed_steps = {0, 1, 2, 3, 5, 10, 15, 20, 30, 40, 49}
        # Match trainer numerics: trainer uses bf16=True which wraps forward
        # in autocast. Without this, our test could pass/fail differently than
        # trainer.train() and the diagnosis is moot.
        _autocast_ctx = (
            torch.autocast(device_type="cuda", dtype=torch.bfloat16)
            if torch.cuda.is_available() else
            torch.amp.autocast(device_type="cpu", dtype=torch.bfloat16)
        )
        print("  running 50 steps on the same batch (bf16 autocast on)...")
        _loop_crashed = False
        _step = -1
        _out = None
        _loss = None
        try:
            for _step in range(50):
                for _p in model.parameters():
                    if _p.grad is not None:
                        _p.grad = None
                with _autocast_ctx:
                    _out = model(**_batch)
                    _loss = _out.loss
                if not torch.isfinite(_loss):
                    print(f"    step {_step}: NON-FINITE LOSS={_loss.item()} — aborting overfit test")
                    break
                _loss.backward()
                _grad_norm = torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], max_norm=10.0
                ).item()
                _test_opt.step()
                _losses.append(_loss.item())
                if _step in _printed_steps:
                    _delta_now = (_body_param.detach() - _body_before).abs().max().item()
                    print(f"    step {_step:2d}: loss={_loss.item():.4f} "
                          f"grad_norm={_grad_norm:.3g} "
                          f"body_max|delta|={_delta_now:.3g}")
        except Exception as _e:
            _loop_crashed = True
            import traceback as _tb
            print(f"\n    !! step {_step} CRASHED: {type(_e).__name__}: {_e}")
            _tb.print_exc()
            print("    !! Likely culprits:")
            print("    !!   - Unsloth grad offload (CPU grads + GPU optimizer = device mismatch)")
            print("    !!   - OOM (snapshot + adam state pushed us over)")
            print("    !!   - dtype mismatch in clip_grad_norm_ / step")
            print("    !! Snapshot restore will still run so model is recoverable.")

        # ---- Verdict ----
        if len(_losses) >= 10:
            _l0, _lN = _losses[0], _losses[-1]
            _drop_pct = 100 * (_l0 - _lN) / max(abs(_l0), 1e-6)
            print(f"\n  loss: {_l0:.4f} -> {_lN:.4f}  ({_drop_pct:.1f}% drop in {len(_losses)} steps)")
        else:
            _drop_pct = float("nan")
            print(f"\n  loss: only {len(_losses)} step(s) ran; no verdict")

        _delta_max = (_body_param.detach() - _body_before).abs().max().item()
        _delta_l2  = (_body_param.detach() - _body_before).float().norm().item()
        print(f"  body {_body_name}:")
        print(f"    max|delta| = {_delta_max:.6g}   (== 0 means optimizer didn't update this param)")
        print(f"    L2 |delta| = {_delta_l2:.6g}")

        # Verify Adam optimizer state actually ticked. In PyTorch 2.x "step"
        # is sometimes a 0-dim tensor and sometimes an int — handle both.
        _opt_state = _test_opt.state.get(_body_param, {})
        _step_val = _opt_state.get("step", 0)
        if hasattr(_step_val, "item"):
            _adam_step = int(_step_val.item())
        else:
            _adam_step = int(_step_val)
        # Also report Adam exp_avg / exp_avg_sq norms — confirms momentum is
        # accumulating, not just step counter ticking.
        _m = _opt_state.get("exp_avg")
        _v = _opt_state.get("exp_avg_sq")
        _m_n = _m.float().norm().item() if _m is not None else float("nan")
        _v_n = _v.float().norm().item() if _v is not None else float("nan")
        print(f"    adam step counter = {_adam_step}  m_norm={_m_n:.4g}  v_norm={_v_n:.4g}")

        _ok = (not _loop_crashed) and (_drop_pct >= 50.0) and (_delta_max > 0) and (_adam_step > 0)
        if not _ok:
            # Print the FULL loss list so user can see the shape
            print(f"\n  FULL LOSS LIST: {[round(l, 4) for l in _losses]}")
            print("\n" + "!" * 60)
            print("OVERFIT TEST FAILED — training mechanism is broken.")
            print("!" * 60)
            print("Loss is not dropping on a single repeated batch. The model")
            print("CAN'T learn anything regardless of LR / epochs. Likely culprits:")
            print("  - Unsloth gradient offload not delivering grads to optimizer")
            print("  - autocast/dtype mismatch between forward and optimizer state")
            print("  - gradient checkpointing reentrancy issue")
            print("  - mean_resizing left embeddings in a degenerate state")

        # ---- Restore base weights so real training starts from a clean slate ----
        # Free Adam state (m+v in fp32 = ~17.7GB for a 2B model) FIRST so the
        # snapshot copy doesn't run with peak memory.
        print("\n  freeing test optimizer state (~17GB Adam m+v)...")
        del _test_opt
        torch.cuda.empty_cache()
        print("  restoring base weights from snapshot...")
        with torch.no_grad():
            for _n, _p in model.named_parameters():
                if _n in _snap:
                    _p.copy_(_snap[_n])
        del _snap, _out, _loss, _losses, _body_before, _it, _batch
        torch.cuda.empty_cache()
        # Also zero any grads left over from the test forward+backward, so the
        # trainer doesn't start with stale grad state.
        for _p in model.parameters():
            if _p.grad is not None:
                _p.grad = None
        print("  restored. (real trainer will start from base model.)")

        if not _ok:
            raise RuntimeError(
                f"Overfit sanity FAILED: loss dropped {_drop_pct:.1f}% in 50 steps "
                f"(need >= 50%), body max|delta|={_delta_max:g}, adam_step={_adam_step}. "
                "Don't waste a 70-minute run on broken training. Debug above first."
            )

        print("\n" + "=" * 60)
        print(f"OVERFIT TEST PASSED: loss dropped {_drop_pct:.1f}%. Training mechanism is alive.")
        print("=" * 60 + "\n")


    print("=" * 60)
    print("RUN CONFIG")
    print("=" * 60)
    print("  dense verdicts : ON (verdict slot after every agent step)")
    print(f"  W_STOP (loss)  : {W_STOP}   (>1 up-weights STOP symbol vs OK)")
    print(f"  LR / epochs    : {LR} / {NUM_TRAIN_EPOCHS}   (effective batch {BATCH_SIZE*GRAD_ACCUM})")
    print(f"  max_seq_len    : {MAX_SEQ_LEN}   (left-truncate keeps tail -> STOP survives)")
    print(f"  debug toggles  : overfit={RUN_OVERFIT_TEST} nan_hooks={INSTALL_NAN_HOOKS} "
          f"verbose_mb={VERBOSE_MICROBATCH} watch_params={WATCH_PARAMS}")
    print("  per-epoch eval : verdict P(STOP) + ppl baseline + reason cosine")
    print(f"  train rows     : {len(hf_ds)}")
    print("=" * 60)
    print("Starting full-FT training (per-epoch eval will fire after each epoch)...")
    t0 = time.time()
    trainer.train()
    print(f"Training + per-epoch eval done in {(time.time()-t0)/60:.1f} min")


## 5. Mode B: load pretrained adapter (optional)

In [ ]:
if USE_PRETRAINED:
    import os
    PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/agent-iam-adapter/submission_adapter"
    required = ["adapter_config.json", "adapter_model.safetensors"]
    for f in required:
        assert os.path.exists(os.path.join(PRETRAINED_ADAPTER_DATASET_PATH, f)), f"missing {f}"
    print("Pretrained adapter OK:", PRETRAINED_ADAPTER_DATASET_PATH)

## 6. Export the full-FT model + tokenizer

In [ ]:
import os

os.makedirs(SUBMISSION_DIR, exist_ok=True)
if TRAIN_ON_KAGGLE:
    # Save the COMPLETE fine-tuned model — full-FT means every parameter
    # is in `model`, so save_pretrained writes the whole checkpoint (no
    # LoRA adapter, no merging required). The resulting directory is
    # directly loadable with:
    #   AutoModelForCausalLM.from_pretrained("/kaggle/working/submission_model")
    #   AutoProcessor.from_pretrained("/kaggle/working/submission_model")
    model.save_pretrained(
        SUBMISSION_DIR,
        safe_serialization=True,   # write model.safetensors, not pytorch_model.bin
        max_shard_size="5GB",      # 2B-bf16 fits in one shard; explicit for clarity
    )
    # Save the tokenizer / processor — must be in the same dir for
    # from_pretrained to round-trip. If unsloth returned a Processor in
    # cell 8 we save that (preserves image_processor + chat_template +
    # tokenizer in one call); otherwise the bare tokenizer.
    if 'processor' in dir() and processor is not None:
        processor.save_pretrained(SUBMISSION_DIR)
    else:
        tokenizer.save_pretrained(SUBMISSION_DIR)

    # Verify the save is complete + load-ready.
    saved = sorted(os.listdir(SUBMISSION_DIR))
    total_bytes = 0
    print(f"\nSaved to {SUBMISSION_DIR}:")
    for f in saved:
        p = os.path.join(SUBMISSION_DIR, f)
        if os.path.isfile(p):
            sz = os.path.getsize(p)
            total_bytes += sz
            print(f"  {f:<40} {sz/1024/1024:>8.1f} MB")
    print(f"  {'TOTAL':<40} {total_bytes/1024/1024:>8.1f} MB")

    required = ["config.json"]
    has_weights = any(f.startswith("model") and (f.endswith(".safetensors") or f == "model.safetensors.index.json")
                      for f in saved)
    has_tok    = any(f.startswith("tokenizer") for f in saved)
    missing = [r for r in required if r not in saved]
    if missing or not has_weights or not has_tok:
        raise RuntimeError(
            f"Incomplete save — missing config={missing} "
            f"weights_present={has_weights} tokenizer_present={has_tok}"
        )
    print("  -> complete: config + weights + tokenizer all present.")
else:
    print("USE_PRETRAINED=1: re-using pre-trained model in", SUBMISSION_DIR)
    for f in sorted(os.listdir(SUBMISSION_DIR)):
        p = os.path.join(SUBMISSION_DIR, f)
        sz = os.path.getsize(p)/1024/1024 if os.path.isfile(p) else 0
        print(f"  {f}: {sz:.1f} MB")


## 7. Eval — Prevention Rate vs False Stop Rate (per epoch) + best-epoch checkpoint

`PerEpochEvalCallback` (registered in cell 12) runs after every epoch:

- Score `data/v0.19/test.jsonl` (auto-discovered in `/kaggle/input`) with the
  current in-memory model wrapped as `TraceMonitor`.
- Write per-epoch scores to `runs-eval/epoch-{N}/scores.jsonl`.
- Score the keyword baseline once (epoch 1 only).
- Compute **PR @ FSR ≤ 1%** on this epoch's scores.
- If it beats the best PR seen so far in this run, save the model to
  `submission_model_best/` (overwriting the previous best) and drop a
  `best_epoch.json` recording which epoch + the metric.

Cell 18 then builds a side-by-side report across all epochs + the keyword
floor, so you can read the PR@1% trajectory and confirm best-epoch choice.

Outputs in `/kaggle/working/`:
- `submission_model/` — **final** epoch weights (always last)
- `submission_model_best/` — **best** epoch weights by PR@FSR=1% + `best_epoch.json`
- `runs-eval/epoch-{1,2,...}/scores.jsonl` — per-epoch per-trace step scores
- `runs-eval/keyword/scores.jsonl` — keyword baseline (one-time)
- `runs-eval/report/report.md` — comparison report across epochs + baseline
- `runs-eval/report/slices.csv` — machine-readable per-slice numbers
- `runs-eval/report/pr_fsr.png` — overlay curve

In [ ]:
# Build the report from per-epoch scores written by PerEpochEvalCallback
# plus the keyword baseline. The table is side-by-side: epoch-1, epoch-2,
# ..., keyword — so you can read PR@1% trajectory across training.
if TRAIN_ON_KAGGLE:
    import glob
    import json
    import os
    from pathlib import Path

    from agent_iam.eval.report import build_report

    EVAL_DIR = "/kaggle/working/runs-eval"

    epoch_files = sorted(
        glob.glob(f"{EVAL_DIR}/epoch-*/scores.jsonl"),
        key=lambda p: int(Path(p).parent.name.split("-")[1]),
    )
    keyword_file = f"{EVAL_DIR}/keyword/scores.jsonl"

    score_files = list(epoch_files)
    if os.path.exists(keyword_file):
        score_files.append(keyword_file)

    if not score_files:
        raise RuntimeError(
            "No scores found under runs-eval/ — did the per-epoch callback fire? "
            "Check that NUM_TRAIN_EPOCHS >= 1 and that the wheel installed in "
            "cell 6 includes agent_iam.eval (rebuild with scripts/build_kaggle_bundle.py)."
        )

    names = [Path(f).parent.name for f in score_files]  # epoch-1, epoch-2, ..., keyword

    summary = build_report(
        score_files,
        out_dir=f"{EVAL_DIR}/report",
        names=names,
        slice_keys=("family", "source", "harm_category"),
    )
    print(json.dumps(summary, indent=2))

    with open(f"{EVAL_DIR}/report/report.md") as f:
        text = f.read()
    print("\n" + "=" * 72)
    cut = text.find("## Per-source")
    print(text if cut < 0 else text[:cut])
    print("=" * 72)

    # Surface which epoch the callback flagged as best.
    best_meta = "/kaggle/working/submission_model_best/best_epoch.json"
    if os.path.exists(best_meta):
        with open(best_meta) as _f:
            bm = json.load(_f)
        print(f"\nBest epoch by PR@FSR=1%: epoch-{bm['epoch']} "
              f"(PR={bm['pr_at_fsr_001']:.3f}, threshold={bm['threshold']})")
        print("  -> weights at /kaggle/working/submission_model_best/")
    else:
        print("\nNo best-epoch checkpoint saved (PR was NaN every epoch?).")

    print("\n/kaggle/working/ contents:")
    for f in sorted(os.listdir("/kaggle/working")):
        p = os.path.join("/kaggle/working", f)
        if os.path.isfile(p):
            sz = os.path.getsize(p) / 1024 / 1024
            print(f"  {f}: {sz:.1f} MB")
        else:
            print(f"  {f}/")
else:
    print("USE_PRETRAINED=1: skipping report build.")
